# Site selection and survey-sensor hand-off (P5)

This notebook runs `identify_reference_sites` and `identify_degraded_sites` for one focal
site, then confirms the shortlist with the survey sensor (TGBS R7). The confirmation runs
through `run_site_timeseries` on `result_to_feature_collection` output, at the survey
sensor's native scale.

It also shows the Dakatcha change terms (baseline vs end-year NDVI, NBR and BSI), the
Dynamic World conversion-pressure term and the CHIRPS rain ratio between the two windows.

Outputs are *proposed* sites for ecologist confirmation and Lead Assessor sign-off,
never final designations.

In [ ]:
from functools import partial
from pathlib import Path

import ee
import numpy as np
import pandas as pd

import eetools
from eetools.compositing import build_period_composites
from eetools.sensors.chirps.preprocessing import get_chirps_collection
from eetools.sensors.dynamicworld.preprocessing import get_dynamic_world_collection
from eetools.sensors.hls.preprocessing import get_hls_merged_collection
from eetools.site_selection import (
    ScreeningTables,
    export_site_selection,
    identify_degraded_sites,
    identify_reference_sites,
    result_to_feature_collection,
)
from eetools.workflows import run_site_timeseries

eetools.initialize(project="charrell-personal")

## 1. Inputs

In [ ]:
FOCAL = Path("data/TGBS_ITF_boundary.gpkg")  # any vector file, or an ee.Geometry
PROJECT, SITE = "TGBS", "ITF"
PROJECT_START_YEAR = 2023  # ITF: started March 2023 -> baseline 2022
END_YEAR = 2025
OUT = Path("outputs") / SITE
SURVEY_SCALE = 30  # HLS native scale (m)
DRIVE_FOLDER = f"{PROJECT}_{SITE}_site_selection"

## 2. Reference sites

Pass `seed_reference_sites` (known references) or `auto_seeds="wdpa_strict"` so the
embedding `agree` bonus is independent evidence. Without either, pseudo-seeds are used and
`agree` is reported only.

In [ ]:
ref = identify_reference_sites(FOCAL, end_year=END_YEAR, auto_seeds="wdpa_strict")
print("rung reached:", ref.params["rung_reached"])
print(*ref.warnings, sep="\n- ")
ref.ranked[["rank", "role", "covers_unit", "rank_score", "intactness", "env_sim",
            "national_class", "wrb_group", "span_note", "mandatory_met"]]

In [ ]:
ref.selection_matrix

## 3. Degraded sites (reusing the reference run's reductions)

The coarse Stages 1-3 are reused from `ref.screening`; only the fine cells and the
signature step cost EECU.

In [ ]:
deg = identify_degraded_sites(
    FOCAL,
    project_start_year=PROJECT_START_YEAR,
    end_year=END_YEAR,
    screening=ref.screening,
)
print(*deg.warnings, sep="\n- ")
deg.ranked[["rank", "rank_score", "degradation", "signature_sim",
            "zero_star_likelihood", "ndvi_performance", "dw_bare", "dw_crops", "dw_built"]]

## 4. Save tables and exports

In [ ]:
ref.screening.to_parquet(OUT / "screening")  # reload with ScreeningTables.from_parquet
export_site_selection(ref, OUT, project=PROJECT, site=SITE)
export_site_selection(deg, OUT, project=PROJECT, site=SITE)

## 5. Survey-sensor confirmation (R7)

One feature collection with the focal site, the ranked references and the ranked
degraded sites, reduced with the HLS builder at its native 30 m scale.

In [ ]:
sites = result_to_feature_collection(ref).merge(
    result_to_feature_collection(deg, include_focal=False)
)
baseline_year = PROJECT_START_YEAR - 1
bands = ["NDVI", "NBR", "BSI"]
builder = partial(get_hls_merged_collection, indices=bands)
stats = run_site_timeseries(
    sites_fc=sites,
    collection_builder=builder,
    bands=bands,
    start_date=f"{baseline_year}-01-01",
    end_date=f"{END_YEAR + 1}-01-01",
    export_folder=DRIVE_FOLDER,
    file_prefix=f"{PROJECT}_{SITE}_hls_annual",
    temporal_scale="annual",
    scale=SURVEY_SCALE,
)

## 6. Dakatcha change terms: baseline vs end year

The difference between the end-year and baseline-year annual medians, per site. BSI change
is min-max normalised over the sites shown, and the BSI-NDVI change correlation is reported
(bare-soil gain should go with greenness loss).

In [ ]:
def annual_site_means(year):
    col = get_hls_merged_collection(
        sites.geometry(), ee.Date(f"{year}-01-01"), ee.Date(f"{year + 1}-01-01"), indices=bands
    )
    img = col.select(bands).median()
    fc = img.reduceRegions(sites, ee.Reducer.mean(), scale=SURVEY_SCALE, tileScale=4)
    rows = [f["properties"] for f in fc.getInfo()["features"]]
    return pd.DataFrame(rows).set_index("site_name")[bands]

base = annual_site_means(baseline_year)
end = annual_site_means(END_YEAR)
change = (end - base).add_prefix("d_")
change["d_BSI_norm"] = (change["d_BSI"] - change["d_BSI"].min()) / (
    change["d_BSI"].max() - change["d_BSI"].min()
)
print("BSI-NDVI change correlation:", round(change["d_BSI"].corr(change["d_NDVI"]), 3))
change

## 7. Dynamic World conversion pressure

Losses in vegetation-class probability (trees, grass, shrub and scrub, flooded vegetation)
plus gains in bare, crops and built, between the baseline and end-year windows.

In [ ]:
VEG = ["trees", "grass", "shrub_and_scrub", "flooded_vegetation"]
CONV = ["bare", "crops", "built"]

def dw_mean(year):
    col = get_dynamic_world_collection(
        sites.geometry(), ee.Date(f"{year}-01-01"), ee.Date(f"{year + 1}-01-01")
    )
    fc = col.select(VEG + CONV).mean().reduceRegions(
        sites, ee.Reducer.mean(), scale=10, tileScale=4
    )
    return pd.DataFrame([f["properties"] for f in fc.getInfo()["features"]]).set_index(
        "site_name"
    )[VEG + CONV]

dw0, dw1 = dw_mean(baseline_year), dw_mean(END_YEAR)
pressure = (dw0[VEG] - dw1[VEG]).clip(lower=0).sum(axis=1) + (
    dw1[CONV] - dw0[CONV]
).clip(lower=0).sum(axis=1)
pressure.rename("dw_conversion_pressure").to_frame()

## 8. CHIRPS rain ratio between the two windows

Use `cadence="pentad"` for whole-year windows (about 5x fewer image reads, with the same
annual totals). Use `"daily"` otherwise, because pentad edges shift a short window by up to
5 days. A ratio outside 0.75-1.33 means rainfall, not condition, may drive the change terms.

In [ ]:
def annual_rain(year, cadence="pentad"):
    col = get_chirps_collection(
        sites.geometry(), f"{year}-01-01", f"{year + 1}-01-01", cadence=cadence
    )
    fc = col.sum().reduceRegions(sites, ee.Reducer.mean(), scale=5566)
    return pd.DataFrame([f["properties"] for f in fc.getInfo()["features"]]).set_index(
        "site_name"
    )["precipitation"]

ratio = (annual_rain(END_YEAR) / annual_rain(baseline_year)).rename("rain_ratio")
if ((ratio < 0.75) | (ratio > 1.33)).any():
    print("WARNING: rain ratio outside 0.75-1.33 for", list(ratio[(ratio < 0.75) | (ratio > 1.33)].index))
ratio.to_frame()